# 02. Data Cleaning: Missing Values and Anomalies

1. Data Loading
2. Definitions and Cleaning Rules
3. Cleaning Steps
   - 3.1 Standardize Missing Values
   - 3.2 Remove Duplicated Month (2001-06)
   - 3.3 Build Song Keys for Missing IDs
   - 3.4 Resolve Same-Month Duplicates
   - 3.5 Remove Records Without Lyrics
   - 3.6 Flag Release-Date Issues
4. Cleaning Summary
5. Export

## 1. Data Loading

### Objective
Load the raw Melon Monthly Chart dataset using the same procedure as `01_data_exploration.ipynb`.

In [1]:
from pathlib import Path
import json
import re
import unicodedata

import pandas as pd

DATA_DIR = Path("../data/raw/melon")
OUTPUT_DIR = Path("../data/interim")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

json_files = sorted(DATA_DIR.rglob("*.json"))

records = []
errors = []

for file in json_files:
    try:
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        info = data.get("info") or []
        chart_info = info[0] if info and isinstance(info[0], dict) else {}

        lyrics = data.get("lyrics") or {}

        records.append({
            "song_id": data.get("song_id"),
            "song_name": data.get("song_name"),
            "artist": data.get("artist"),
            "album": data.get("album"),
            "release_date": data.get("release_date"),
            "genre": data.get("genre"),
            "chart_year": chart_info.get("year"),
            "chart_month": chart_info.get("month"),
            "chart_rank": chart_info.get("rank"),
            "lyrics": lyrics.get("lines") if isinstance(lyrics, dict) else None,
            "source_file": str(file.relative_to(DATA_DIR))
        })

    except Exception as e:
        errors.append((str(file), str(e)))

df_raw = pd.DataFrame(records)

print(f"Total records: {len(df_raw):,}")
print(f"Read errors: {len(errors):,}")

Total records: 25,876
Read errors: 0


In [2]:
# Cleaning log: one row per step
cleaning_log = []

def log_step(step, action, affected, data):
    cleaning_log.append({
        "step": step,
        "action": action,
        "records_affected": affected,
        "records_remaining": len(data),
    })
    print(f"[{step}] {action}: {affected:,} records affected, {len(data):,} remaining")

df = df_raw.copy()
log_step("1", "Load raw records", 0, df)

[1] Load raw records: 0 records affected, 25,876 remaining


## 2. Definitions and Cleaning Rules

### Missing Values
| Issue | Definition | Scope (EDA) | Rule |
|---|---|---|---|
| Missing metadata | Null, empty, or whitespace-only `song_id`, `release_date`, or `genre` | 348 records | Standardize to NA (3.1) |
| Missing song ID | No valid `song_id` | 346 records | Keep; assign a song key (3.3) |
| Missing release date | No valid `release_date` | 348 records | Keep; flag (3.6) |
| Missing lyrics | `lyrics` is not a list or has no non-empty line | 640 records | Remove (3.5) |
| Missing chart months | Months with 0 records | 2014-06 to 2014-09, 2023-03 | Not recoverable; reported as a limitation |

### Anomalies
| Issue | Definition | Scope (EDA) | Rule |
|---|---|---|---|
| Duplicated month | Song list identical to another month | 2001-06 (copy of 2001-08) | Remove the month (3.2) |
| Same-month duplicate | Same song appears more than once in one chart month | 6 groups (12 records) | Keep the record with the highest rank (3.4) |
| Release after chart | Release date later than the end of the chart month | 214 records | Keep; flag (3.6) |
| Chart discontinuity | Overlap with previous month < 0.2 | 2004-11 | Keep; reported as a limitation |
| Post-gap month | Previous month missing, so low overlap is expected | 2014-10 | Keep; false positive |

## 3. Cleaning Steps

### 3.1 Standardize Missing Values

#### Objective
Represent all missing metadata as NA, including empty strings that standard null checks do not detect (EDA 3.3).

#### Procedure
- Convert empty and whitespace-only strings in text columns to NA.
- Strip leading and trailing whitespace from the remaining values.

In [3]:
text_columns = ["song_id", "song_name", "artist", "album", "release_date", "genre"]

before_mask = df[text_columns].isna()
before_na = before_mask.sum()

for col in text_columns:
    df[col] = df[col].str.strip().replace("", pd.NA)

after_na = df[text_columns].isna().sum()

display(pd.DataFrame({
    "NA before": before_na,
    "NA after": after_na,
    "Converted": after_na - before_na,
}))

n_records = int((df[text_columns].isna() & ~before_mask).any(axis=1).sum())
log_step("3.1", "Standardize empty strings to NA", n_records, df)

,NA before,NA after,Converted
song_id,346,346,0
song_name,0,0,0
artist,0,0,0
album,0,0,0
release_date,0,348,348
genre,0,348,348


[3.1] Standardize empty strings to NA: 348 records affected, 25,876 remaining


### 3.2 Remove Duplicated Month (2001-06)

#### Objective
Remove the 2001-06 chart, which contains a copy of the 2001-08 chart (EDA 4.9).

#### Procedure
- Remove all records with `chart_year == 2001` and `chart_month == 6`.
- The original June 2001 chart is not available in the dataset.

In [4]:
is_dup_month = (df["chart_year"] == 2001) & (df["chart_month"] == 6)

n_removed = int(is_dup_month.sum())
df = df[~is_dup_month].reset_index(drop=True)

log_step("3.2", "Remove duplicated month 2001-06", n_removed, df)

[3.2] Remove duplicated month 2001-06: 64 records affected, 25,812 remaining


### 3.3 Build Song Keys for Missing IDs

#### Objective
Assign every record a `song_key` so that records with missing song IDs can be kept and deduplicated later.

#### Procedure
- Normalize song names and artists (Unicode NFKC, lowercase, collapsed whitespace).
- Records with a valid song ID: `song_key = song_id`.
- Records without a song ID:
  - If the normalized name and artist match exactly one existing song ID, reuse that ID.
  - Otherwise, assign a new key `noid_<n>` shared by records with the same normalized name and artist.
- Parentheses are kept, so remixes and special versions remain separate songs.

In [5]:
def normalize_text(s):
    if pd.isna(s):
        return ""
    s = unicodedata.normalize("NFKC", s).lower()
    return re.sub(r"\s+", " ", s).strip()

df["name_norm"] = df["song_name"].map(normalize_text)
df["artist_norm"] = df["artist"].map(normalize_text)

has_id = df["song_id"].notna()

# Name + artist pairs that map to exactly one song ID
id_lookup = (
    df[has_id]
    .groupby(["name_norm", "artist_norm"])["song_id"]
    .agg(lambda x: x.iloc[0] if x.nunique() == 1 else pd.NA)
    .dropna()
)

df["song_key"] = df["song_id"]
df["key_source"] = pd.Series("song_id", index=df.index).where(has_id)

missing = df[~has_id]
matched = pd.Series(
    [id_lookup.get((n, a), pd.NA) for n, a in zip(missing["name_norm"], missing["artist_norm"])],
    index=missing.index, dtype="object",
)
df.loc[matched.dropna().index, "song_key"] = matched.dropna()
df.loc[matched.dropna().index, "key_source"] = "matched_id"

# Remaining records: new key per name + artist pair
unmatched = df["song_key"].isna()
pair_codes = (
    df.loc[unmatched]
    .groupby(["name_norm", "artist_norm"], sort=True)
    .ngroup()
)
df.loc[unmatched, "song_key"] = "noid_" + pair_codes.astype(str)
df.loc[unmatched, "key_source"] = "new_key"

print("Song key sources:")
display(df["key_source"].value_counts().to_frame("records"))

print("\nUnique songs among missing-ID records:")
print(f"Matched to existing IDs: {df.loc[df.key_source == 'matched_id', 'song_key'].nunique():,}")
print(f"New keys: {df.loc[df.key_source == 'new_key', 'song_key'].nunique():,}")

print("\nSample of new keys:")
display(
    df[df.key_source == "new_key"]
    .drop_duplicates("song_key")
    [["song_key", "song_name", "artist", "chart_year", "chart_month"]]
    .head(15)
)

log_step("3.3", "Assign song keys to missing-ID records", int((~has_id).sum()), df)

Song key sources:


,records
key_source,
song_id,25467
new_key,344
matched_id,1



Unique songs among missing-ID records:
Matched to existing IDs: 1
New keys: 134

Sample of new keys:


,song_key,song_name,artist,chart_year,chart_month
15,noid_76,겨울이야기,DJ DOC,2000,1
26,noid_53,Say You Will,유승준,2000,1
71,noid_111,상처 (Special Re-Mix),조성모,2000,1
110,noid_115,약속,유영석,2000,2
171,noid_35,Love,윤현석,2000,3
192,noid_94,마린 일병 구하기(Remix),허니 패밀리,2000,3
239,noid_69,Vision,유승준,2000,4
415,noid_104,부탁,김원준,2000,7
449,noid_77,결혼 행진곡,Felix Mendelssohn,2000,7
501,noid_27,Hey Girl,TJ,2000,8


[3.3] Assign song keys to missing-ID records: 345 records affected, 25,812 remaining


### 3.4 Resolve Same-Month Duplicates

#### Objective
Ensure each song appears at most once per chart month (EDA 3.6).

#### Procedure
- Identify records sharing the same `song_key`, `chart_year`, and `chart_month`.
- Keep the record with the highest chart rank (smallest `chart_rank`) in each group.
- Groups may include missing-ID records matched to an existing ID in 3.3, so the count can exceed the 6 groups found in EDA.

In [6]:
month_keys = ["song_key", "chart_year", "chart_month"]

dup_groups = df[df.duplicated(month_keys, keep=False)].sort_values(month_keys + ["chart_rank"])
print(f"Duplicate groups: {dup_groups.groupby(month_keys).ngroups}")
display(dup_groups[["song_key", "key_source", "song_name", "artist", "chart_year", "chart_month", "chart_rank"]])

before = len(df)
df = (
    df.sort_values(["chart_year", "chart_month", "chart_rank"])
      .drop_duplicates(month_keys, keep="first")
      .reset_index(drop=True)
)

log_step("3.4", "Keep highest-ranked record per song and month", before - len(df), df)

Duplicate groups: 6


,song_key,key_source,song_name,artist,chart_year,chart_month,chart_rank
2486,317027,song_id,라라라,이수영,2003,5,36
2502,317027,song_id,라라라,이수영,2003,5,50
471,4077,song_id,오! 가니,컨츄리 꼬꼬,2000,8,18
494,4077,song_id,오! 가니,컨츄리 꼬꼬,2000,8,41
248,494157,song_id,다 알아요,양파,2000,4,49
263,494157,song_id,다 알아요,양파,2000,4,50
893,84375,song_id,꿈꾸는 소녀,YB (윤도현 밴드),2001,2,50
898,84375,song_id,꿈꾸는 소녀,YB (윤도현 밴드),2001,2,50
325,85257,song_id,바꿔,이정현,2000,5,50
333,85257,song_id,바꿔,이정현,2000,5,50


[3.4] Keep highest-ranked record per song and month: 6 records affected, 25,806 remaining


### 3.5 Remove Records Without Lyrics

#### Objective
Remove records that cannot be used for lyrics analysis (EDA 3.7).

#### Procedure
- A record has valid lyrics if `lyrics` is a list with at least one non-empty line.
- Remove records without valid lyrics.

In [7]:
def has_valid_lyrics(lines):
    return (
        isinstance(lines, list)
        and any(isinstance(line, str) and line.strip() for line in lines)
    )

no_lyrics = ~df["lyrics"].apply(has_valid_lyrics)

print("Records without lyrics by year:")
display(df[no_lyrics].groupby("chart_year").size().to_frame("records"))

n_removed = int(no_lyrics.sum())
df = df[~no_lyrics].reset_index(drop=True)

df["lyric_line_count"] = df["lyrics"].apply(
    lambda x: sum(1 for line in x if isinstance(line, str) and line.strip())
)

log_step("3.5", "Remove records without lyrics", n_removed, df)

Records without lyrics by year:


,records
chart_year,
2000,48
2001,37
2002,16
2003,17
2004,30
2005,35
2006,37
2007,26
2008,16


[3.5] Remove records without lyrics: 639 records affected, 25,167 remaining


### 3.6 Flag Release-Date Issues

#### Objective
Mark records with missing or inconsistent release dates without removing them (EDA 4.3).

#### Procedure
- Parse release dates (`YYYY.MM.DD` and `YYYY.MM`).
- `flag_release_missing`: release date is missing or cannot be parsed.
- `flag_release_after_chart`: release date is later than the last day of the chart month.
- These flags are used to correct the time variable in the song-level dataset (first chart entry year replaces flagged release years).

In [8]:
df["release_date_parsed"] = pd.to_datetime(df["release_date"], format="mixed", errors="coerce")

chart_month_end = (
    pd.to_datetime(dict(year=df["chart_year"], month=df["chart_month"], day=1))
    + pd.offsets.MonthEnd(0)
)

df["flag_release_missing"] = df["release_date_parsed"].isna()
df["flag_release_after_chart"] = df["release_date_parsed"] > chart_month_end

display(pd.DataFrame({
    "records": [df["flag_release_missing"].sum(), df["flag_release_after_chart"].sum()],
    "songs": [
        df.loc[df["flag_release_missing"], "song_key"].nunique(),
        df.loc[df["flag_release_after_chart"], "song_key"].nunique(),
    ],
}, index=["flag_release_missing", "flag_release_after_chart"]))

n_flagged = int((df["flag_release_missing"] | df["flag_release_after_chart"]).sum())
log_step("3.6", "Flag missing or inconsistent release dates", n_flagged, df)

,records,songs
flag_release_missing,10,3
flag_release_after_chart,186,89


[3.6] Flag missing or inconsistent release dates: 196 records affected, 25,167 remaining


## 4. Cleaning Summary

In [9]:
log_df = pd.DataFrame(cleaning_log)
display(log_df)

print(f"Raw records: {len(df_raw):,}")
print(f"Clean records: {len(df):,} ({len(df) / len(df_raw):.2%} retained)")
print(f"Unique songs (song_key): {df['song_key'].nunique():,}")

,step,action,records_affected,records_remaining
0,1,Load raw records,0,25876
1,3.1,Standardize empty strings to NA,348,25876
2,3.2,Remove duplicated month 2001-06,64,25812
3,3.3,Assign song keys to missing-ID records,345,25812
4,3.4,Keep highest-ranked record per song and month,6,25806
5,3.5,Remove records without lyrics,639,25167
6,3.6,Flag missing or inconsistent release dates,196,25167


Raw records: 25,876
Clean records: 25,167 (97.26% retained)
Unique songs (song_key): 7,988


In [10]:
# Records per year before and after cleaning
yearly = pd.DataFrame({
    "raw": df_raw.groupby("chart_year").size(),
    "clean": df.groupby("chart_year").size(),
}).fillna(0).astype(int)
yearly["removed"] = yearly["raw"] - yearly["clean"]
yearly["retained (%)"] = (yearly["clean"] / yearly["raw"] * 100).round(2)

display(yearly)

,raw,clean,removed,retained (%)
chart_year,,,,
2000,773,722,51,93.40
2001,758,655,103,86.41
2002,728,712,16,97.80
2003,755,737,18,97.62
2004,836,806,30,96.41
2005,1160,1125,35,96.98
2006,1189,1152,37,96.89
2007,1196,1170,26,97.83
2008,1194,1178,16,98.66


## 5. Export

### Output
- `data/interim/chart_records_clean.pkl`: cleaned chart-level records (lyrics kept as lists)
- `data/interim/cleaning_log.csv`: cleaning steps and record counts

In [11]:
clean_columns = [
    "song_key", "key_source", "song_id", "song_name", "artist", "album",
    "release_date", "release_date_parsed", "genre",
    "chart_year", "chart_month", "chart_rank",
    "lyrics", "lyric_line_count",
    "flag_release_missing", "flag_release_after_chart",
    "source_file",
]

df_clean = df[clean_columns].copy()

df_clean.to_pickle(OUTPUT_DIR / "chart_records_clean.pkl")
log_df.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")

print(f"Saved {len(df_clean):,} records to {OUTPUT_DIR / 'chart_records_clean.pkl'}")
print(f"Saved cleaning log to {OUTPUT_DIR / 'cleaning_log.csv'}")

Saved 25,167 records to ../data/interim/chart_records_clean.pkl
Saved cleaning log to ../data/interim/cleaning_log.csv
